# Phase 8 — Spark UI, Monitoring & Debugging Experiments Notebook

This is the **worked SOLUTION notebook** for Phase 8.

Run it **top-to-bottom** while keeping the notebook kernel alive so the live Spark UI remains available.

Every experiment follows:

```text
predict
    ↓
run one known action
    ↓
job
    ↓
stage
    ↓
tasks
    ↓
record ACTUAL runtime evidence
    ↓
connect to df.explain('formatted')
    ↓
root cause
    ↓
change ONE thing when justified
    ↓
rerun
    ↓
compare
    ↓
reconcile correctness
```

Core question:

> **Where is the time or failure occurring, and what physical behavior caused it?**

Important:

- The code and reasoning framework are worked solutions.
- Spark UI metrics are **not pre-filled**, because they depend on your runtime.
- Record what Spark actually shows for job IDs, stage IDs, task durations, shuffle sizes, spill, failures, executor behavior, and AQE changes.
- This notebook does not perform the formal mastery gate, update `ROADMAP.md`, mark Phase 8 complete, or enter Phase 9.


<a id="toc"></a>
## Table of Contents

- [Setup and Practice Data](#setup-and-practice-data)
- [Experiment Protocol](#experiment-protocol)
- [Experiment 1 — Multiple Actions and Jobs](#experiment-1)
- [Experiment 2 — Stage Boundaries and Shuffle Evidence](#experiment-2)
- [Experiment 3 — Balanced Tasks vs. Skewed Stragglers](#experiment-3)
- [Experiment 4 — Too Few vs. Too Many Tasks](#experiment-4)
- [Experiment 5 — Sort-Merge vs. Broadcast Join](#experiment-5)
- [Experiment 6 — Spill and Memory Pressure](#experiment-6)
- [Experiment 7 — Failed Task Localization](#experiment-7)
- [Experiment 8 — Executor Utilization and Driver Misuse](#experiment-8)
- [Experiment 9 — AQE Runtime Behavior](#experiment-9)
- [Experiment 10 — Recomputed Lineage vs. Cache Reuse](#experiment-10)
- [Applied Phase 8 Project](#applied-project)
- [Cleanup](#cleanup)

---

<a id="setup-and-practice-data"></a>
# Setup and Practice Data

Main grains:

```text
fact_sales_df
= one row per sale_id

dim_store_df
= one row per store_id

dim_product_df
= one row per product_id

balanced_sales_df / skewed_sales_df
= one row per sale_id
```


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


spark = (
    SparkSession.builder
    .appName('phase_08_spark_ui_experiments')
    .master('local[4]')
    # Keep ordinary experiments static so stage/task behavior is easier to compare.
    .config('spark.sql.shuffle.partitions', '12')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

print(f'Live Spark UI: {spark.sparkContext.uiWebUrl}')
print('Keep this notebook kernel running while you inspect completed jobs/stages.')


In [ ]:
# Grain: one row per sale_id.
fact_sales_df = (
    spark.range(
        start=0,
        end=240000,
        step=1,
        numPartitions=12,
    )
    .select(
        F.col('id').cast('long').alias('sale_id'),
        F.date_add(
            F.lit('2026-01-01').cast('date'),
            (F.col('id') % 180).cast('int'),
        ).alias('order_date'),
        F.concat(
            F.lit('S'),
            F.lpad(
                ((F.col('id') % 40) + F.lit(1)).cast('string'),
                3,
                '0',
            ),
        ).alias('store_id'),
        F.concat(
            F.lit('P'),
            F.lpad(
                ((F.col('id') % 500) + F.lit(1)).cast('string'),
                4,
                '0',
            ),
        ).alias('product_id'),
        F.concat(
            F.lit('C'),
            F.lpad(
                ((F.col('id') % 10000) + F.lit(1)).cast('string'),
                5,
                '0',
            ),
        ).alias('customer_id'),
        F.when(
            (F.col('id') % 10) < 8,
            F.lit('COMPLETED'),
        )
        .when(
            (F.col('id') % 10) == 8,
            F.lit('CANCELLED'),
        )
        .otherwise(F.lit('RETURNED'))
        .alias('order_status'),
        ((F.col('id') % 5) + F.lit(1)).cast('int').alias('quantity'),
        (
            F.lit(5.00)
            + ((F.col('id') % 75) * F.lit(0.75))
        )
        .cast(DecimalType(12, 2))
        .alias('unit_price'),
    )
    .withColumn(
        'gross_sales',
        (F.col('quantity') * F.col('unit_price'))
        .cast(DecimalType(16, 2)),
    )
    .withColumn('year', F.year('order_date'))
    .withColumn('month', F.month('order_date'))
)


# Grain: one row per store_id.
dim_store_df = (
    spark.range(
        start=1,
        end=41,
        step=1,
        numPartitions=2,
    )
    .select(
        F.concat(
            F.lit('S'),
            F.lpad(F.col('id').cast('string'), 3, '0'),
        ).alias('store_id'),
        F.concat(
            F.lit('Store '),
            F.col('id').cast('string'),
        ).alias('store_name'),
        F.when(F.col('id') <= 20, F.lit('ON'))
        .otherwise(F.lit('BC'))
        .alias('province'),
        F.when((F.col('id') % 2) == 0, F.lit('URBAN'))
        .otherwise(F.lit('SUBURBAN'))
        .alias('store_format'),
    )
)


# Grain: one row per product_id.
dim_product_df = (
    spark.range(
        start=1,
        end=501,
        step=1,
        numPartitions=4,
    )
    .select(
        F.concat(
            F.lit('P'),
            F.lpad(F.col('id').cast('string'), 4, '0'),
        ).alias('product_id'),
        F.concat(
            F.lit('Product '),
            F.col('id').cast('string'),
        ).alias('product_name'),
        F.concat(
            F.lit('CATEGORY_'),
            (F.col('id') % 12).cast('string'),
        ).alias('category'),
    )
)


# Grain: one row per sale_id; key frequencies are intentionally balanced.
balanced_sales_df = (
    spark.range(
        start=0,
        end=240000,
        step=1,
        numPartitions=12,
    )
    .select(
        F.col('id').cast('long').alias('sale_id'),
        F.concat(
            F.lit('S'),
            F.lpad(
                ((F.col('id') % 40) + F.lit(1)).cast('string'),
                3,
                '0',
            ),
        ).alias('store_id'),
        F.lit(1).cast('long').alias('units'),
    )
)


# Grain: one row per sale_id; S001 deliberately owns about 80% of rows.
skewed_sales_df = (
    spark.range(
        start=0,
        end=240000,
        step=1,
        numPartitions=12,
    )
    .select(
        F.col('id').cast('long').alias('sale_id'),
        F.when(
            (F.col('id') % 100) < 80,
            F.lit('S001'),
        )
        .otherwise(
            F.concat(
                F.lit('S'),
                F.lpad(
                    ((F.col('id') % 39) + F.lit(2)).cast('string'),
                    3,
                    '0',
                ),
            )
        )
        .alias('store_id'),
        F.lit(1).cast('long').alias('units'),
    )
)


In [ ]:
def run_action(label, action):
    '''Run one labelled materializing action and return result + elapsed seconds.'''

    # Make the action easier to identify in the Jobs view.
    spark.sparkContext.setJobDescription(label)

    started_at = perf_counter()
    result = action()
    elapsed_seconds = perf_counter() - started_at

    print(f'{label}: {elapsed_seconds:.3f} seconds')
    print('Treat wall-clock time as supporting evidence, not proof by itself.')

    return result, elapsed_seconds


def show_key_frequency(df, key_column, label):
    '''Show a small aggregated key-frequency diagnostic.'''

    print(f'\n{label}')

    (
        df
        .groupBy(key_column)
        .agg(F.count('*').alias('row_count'))
        .orderBy(
            F.col('row_count').desc(),
            F.col(key_column).asc_nulls_last(),
        )
        .show(20, truncate=False)
    )


def reconcile_scalar(left_df, right_df, measure_column, label):
    '''Assert that one aggregate business measure is identical.'''

    left_value = (
        left_df
        .agg(F.sum(measure_column).alias('measure'))
        .first()['measure']
    )

    right_value = (
        right_df
        .agg(F.sum(measure_column).alias('measure'))
        .first()['measure']
    )

    print(f'{label}: {left_value} == {right_value}')
    assert left_value == right_value


def assert_unique_key(df, key_column):
    '''Assert uniqueness for a dimension/business key.'''

    duplicate_count = (
        df
        .groupBy(key_column)
        .count()
        .filter(F.col('count') > 1)
        .count()
    )

    assert duplicate_count == 0


In [ ]:
# Freeze core correctness before performance experiments.
assert_unique_key(dim_store_df, 'store_id')
assert_unique_key(dim_product_df, 'product_id')

fact_row_count = fact_sales_df.count()
distinct_sale_id_count = fact_sales_df.select('sale_id').distinct().count()

assert distinct_sale_id_count == fact_row_count

print('Core grain / key checks passed.')


[Back to Table of Contents](#toc)

---

<a id="experiment-protocol"></a>
# Experiment Protocol

For every experiment:

1. State the required grain and correctness invariant.
2. Predict the physical behavior.
3. Inspect `df.explain('formatted')`.
4. Run one labelled materializing action.
5. Locate the corresponding job/query.
6. Find the slow/failing stage.
7. Compare task durations and data sizes.
8. Record actual shuffle/spill/failure/executor evidence.
9. Connect runtime evidence to the physical operator.
10. Change **one thing** only when evidence justifies it.
11. Rerun the same action.
12. Reconcile correctness.

Use this runtime record:

```text
RUN LABEL:
MATERIALIZING ACTION:
PREDICTION:

JOB / QUERY
- job ID(s):
- query ID:
- elapsed time:

STAGE
- stage ID:
- role:
- task count:
- stage duration:
- input:
- shuffle write:
- shuffle read:
- memory spill:
- disk spill:
- failed/retried tasks:

TASKS
- typical duration:
- maximum duration:
- typical data size:
- maximum data size:
- stragglers?:

EXECUTORS
- useful parallelism?:
- repeated slow/failing executor?:
- notable GC/spill?:

PLAN
- relevant operators:
- Exchange(s):
- join strategy:
- AQE evidence:

ROOT-CAUSE HYPOTHESIS:
```

**Never fill a runtime field from expectation. Record what the UI actually shows.**


[Back to Table of Contents](#toc)

---

<a id="experiment-1"></a>
# Experiment 1 — Multiple Actions and Jobs

## Question

Which action triggered the runtime work, and does a second action over the same uncached lineage recompute upstream work?

### Worked prediction

`groupBy('store_id')` requires rows with the same key to be colocated, so an `Exchange` is expected.

The two actions are separate materializations. Without persistence, do **not** assume that Action 2 automatically reuses Action 1's computed result.

The exact number of jobs/stages must come from your run.


In [ ]:
completed_store_sales_df = (
    fact_sales_df
    .filter(F.col('order_status') == 'COMPLETED')
    .groupBy('store_id')
    .agg(
        F.sum('gross_sales').alias('gross_sales'),
        F.count('*').alias('sale_line_count'),
    )
)

completed_store_sales_df.explain('formatted')


In [ ]:
completed_store_count, experiment_1_count_seconds = run_action(
    'PHASE 8 - EXP 1 - completed store sales count',
    completed_store_sales_df.count,
)

print(f'completed store rows: {completed_store_count}')


In [ ]:
completed_store_total, experiment_1_total_seconds = run_action(
    'PHASE 8 - EXP 1 - completed store sales total',
    lambda: (
        completed_store_sales_df
        .agg(F.sum('gross_sales').alias('gross_sales'))
        .first()['gross_sales']
    ),
)

print(f'completed gross sales: {completed_store_total}')


### Record actual evidence

```text
ACTION 1
- job ID(s):
- stage count:
- slowest stage:
- shuffle write/read:

ACTION 2
- job ID(s):
- stage count:
- slowest stage:
- shuffle write/read:

Does equivalent upstream aggregation work appear again?:
```

### Worked reasoning

A DataFrame is a lazy logical description. Executing one action does not normally turn its result into a persisted reusable dataset.

If the UI shows the expensive lineage again for Action 2, that is runtime evidence of recomputation.

Do not cache yet; Experiment 10 tests whether persistence is justified.


[Back to Table of Contents](#toc)

---

<a id="experiment-2"></a>
# Experiment 2 — Stage Boundaries and Shuffle Evidence

## Question

Which stage is expensive, and what physical operator created that work?

### Worked prediction

The final `groupBy('province')` requires redistribution by `province`.

The join strategy is planner-dependent unless forced, so inspect the plan rather than assuming broadcast or sort-merge behavior.


In [ ]:
province_sales_df = (
    fact_sales_df
    .filter(F.col('order_status') == 'COMPLETED')
    .select(
        'store_id',
        'gross_sales',
    )
    .join(
        dim_store_df.select(
            'store_id',
            'province',
        ),
        on='store_id',
        how='inner',
    )
    .groupBy('province')
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

province_sales_df.explain('formatted')


In [ ]:
province_sales_rows, experiment_2_seconds = run_action(
    'PHASE 8 - EXP 2 - province sales collect',
    lambda: province_sales_df.orderBy('province').collect(),
)

for row in province_sales_rows:
    print(row)


### Record actual evidence

```text
job/query:
source/input stage:
stage with shuffle write:
stage with shuffle read:
stage with final aggregation:
slowest stage:
task count in slowest stage:
typical task duration:
max task duration:
```

### Worked reasoning template

```text
Stage ___ is the runtime location.
The physical plan shows ___.
That operator requires ___ physical behavior.
The stage's tasks show ___.
Therefore the evidence supports ___ as the cause of the runtime cost.
```

A stage ID alone is not a diagnosis.


[Back to Table of Contents](#toc)

---

<a id="experiment-3"></a>
# Experiment 3 — Balanced Tasks vs. Skewed Stragglers

## Question

Does a verified hot key create measurably uneven task work?

### Worked prediction

Balanced keys should have a better chance of producing similarly sized reduce-side work.

For the skewed relation, `S001` owns about 80% of rows. If hash partitioning by `store_id` materially exposes that skew, one or a few reduce tasks may consume much more shuffle data and become stragglers.

The actual effect must be observed.


In [ ]:
show_key_frequency(
    balanced_sales_df,
    'store_id',
    'BALANCED KEY FREQUENCY',
)

balanced_store_units_df = (
    balanced_sales_df
    .groupBy('store_id')
    .agg(F.sum('units').alias('units'))
)

balanced_store_units_df.explain('formatted')

_, balanced_seconds = run_action(
    'PHASE 8 - EXP 3 - balanced aggregation',
    balanced_store_units_df.count,
)


In [ ]:
show_key_frequency(
    skewed_sales_df,
    'store_id',
    'SKEWED KEY FREQUENCY',
)

skewed_store_units_df = (
    skewed_sales_df
    .groupBy('store_id')
    .agg(F.sum('units').alias('units'))
)

skewed_store_units_df.explain('formatted')

_, skewed_seconds = run_action(
    'PHASE 8 - EXP 3 - skewed aggregation',
    skewed_store_units_df.count,
)


### Record actual comparison

| Evidence | Balanced | Skewed |
|---|---:|---:|
| Reduce-stage task count |  |  |
| Typical task duration |  |  |
| Max task duration |  |  |
| Typical shuffle read |  |  |
| Max shuffle read |  |  |
| Memory spill |  |  |
| Disk spill |  |  |

### Worked diagnosis rule

Do **not** call a slow task skew merely because it is slow.

Stronger evidence is:

```text
verified hot key
+
Exchange on that key
+
slow task has disproportionately large shuffle/input volume
+
possibly concentrated spill
```

If task sizes remain similar, investigate another cause rather than forcing the skew conclusion.


[Back to Table of Contents](#toc)

---

<a id="experiment-4"></a>
# Experiment 4 — Too Few vs. Too Many Tasks

## Question

Is parallelism limited by too few partitions, or is useful work fragmented into too many tiny tasks?

### Worked prediction

`repartition(2)` should create two downstream partitions after its Exchange.

`repartition(240)` deliberately creates many small downstream partitions.

Neither number is universally bad. Judge the actual task sizes and durations.


In [ ]:
base_task_df = spark.range(
    start=0,
    end=360000,
    step=1,
    numPartitions=12,
)

few_partitions_df = (
    base_task_df
    .repartition(2)
    .select(
        F.col('id'),
        (F.col('id') % 1000).alias('group_id'),
    )
)

print(f'few_partitions_df partitions: {few_partitions_df.rdd.getNumPartitions()}')
few_partitions_df.explain('formatted')

_, few_partition_seconds = run_action(
    'PHASE 8 - EXP 4 - two partitions',
    few_partitions_df.count,
)


In [ ]:
many_partitions_df = (
    base_task_df
    .repartition(240)
    .select(
        F.col('id'),
        (F.col('id') % 1000).alias('group_id'),
    )
)

print(f'many_partitions_df partitions: {many_partitions_df.rdd.getNumPartitions()}')
many_partitions_df.explain('formatted')

_, many_partition_seconds = run_action(
    'PHASE 8 - EXP 4 - 240 partitions',
    many_partitions_df.count,
)


### Record actual comparison

```text
TWO PARTITIONS
- relevant stage tasks:
- typical data/task:
- typical task duration:
- executor capacity left idle?:

240 PARTITIONS
- relevant stage tasks:
- typical data/task:
- typical task duration:
- tasks extremely short?:
```

### Worked reasoning

If only two tasks are runnable, adding more executor capacity cannot create more parallel work for that stage.

If hundreds of tasks process tiny amounts of data, scheduling overhead may become disproportionate.

Remember:

```text
partition count
!=
partition balance
```


[Back to Table of Contents](#toc)

---

<a id="experiment-5"></a>
# Experiment 5 — Sort-Merge vs. Broadcast Join

## Question

How does join strategy change the shuffle behavior visible in the UI?

### Correctness precondition

`dim_store_df` is unique on `store_id`, so the dimension join preserves fact grain before aggregation.

### Worked hypothesis

The forced sort-merge baseline should require repartitioning/sorting for the join.

Broadcasting the validated-small store dimension should remove the ordinary fact-side join-key shuffle for that join.

The final province aggregation can still require its own shuffle.


In [ ]:
assert_unique_key(dim_store_df, 'store_id')

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

sort_merge_province_sales_df = (
    fact_sales_df
    .filter(F.col('order_status') == 'COMPLETED')
    .select(
        'store_id',
        'gross_sales',
    )
    .hint('merge')
    .join(
        dim_store_df
        .select(
            'store_id',
            'province',
        )
        .hint('merge'),
        on='store_id',
        how='inner',
    )
    .groupBy('province')
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

sort_merge_province_sales_df.explain('formatted')

_, sort_merge_seconds = run_action(
    'PHASE 8 - EXP 5 - forced sort merge',
    sort_merge_province_sales_df.count,
)


In [ ]:
# Change ONE thing: replicate the validated-small store dimension.
broadcast_province_sales_df = (
    fact_sales_df
    .filter(F.col('order_status') == 'COMPLETED')
    .select(
        'store_id',
        'gross_sales',
    )
    .join(
        F.broadcast(
            dim_store_df.select(
                'store_id',
                'province',
            )
        ),
        on='store_id',
        how='inner',
    )
    .groupBy('province')
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

broadcast_province_sales_df.explain('formatted')

_, broadcast_seconds = run_action(
    'PHASE 8 - EXP 5 - broadcast store dimension',
    broadcast_province_sales_df.count,
)


In [ ]:
# Optimization is valid only if the business result still reconciles.
reconcile_scalar(
    sort_merge_province_sales_df,
    broadcast_province_sales_df,
    'gross_sales',
    'sort merge vs. broadcast province sales',
)

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10485760')


### Record actual comparison

| Evidence | Sort-merge baseline | Broadcast version |
|---|---:|---:|
| Join operator |  |  |
| Join-related Exchange(s) |  |  |
| Shuffle write |  |  |
| Shuffle read |  |  |
| Slowest relevant stage |  |  |
| Typical task duration |  |  |
| Max task duration |  |  |
| Spill |  |  |

### Worked explanation

A valid causal conclusion is:

```text
The physical plan changed from SortMergeJoin to BroadcastHashJoin.
The runtime evidence showed that ordinary join-key redistribution was removed
or reduced as expected.
The final aggregation still required redistribution by province.
The business result reconciled exactly.
```

Use that conclusion only if your plan/UI support it.


[Back to Table of Contents](#toc)

---

<a id="experiment-6"></a>
# Experiment 6 — Spill and Memory Pressure

## Question

If spill appears, which tasks spill and why?

### Worked prediction

This aggregation has many grouping keys and relatively few input partitions, so it is a reasonable local **spill candidate**.

It does not guarantee spill. A zero-spill result is valid runtime evidence.


In [ ]:
spill_candidate_df = (
    spark.range(
        start=0,
        end=600000,
        step=1,
        numPartitions=4,
    )
    .select(
        (F.col('id') % 200000).alias('group_id'),
        F.col('id').cast('double').alias('value'),
    )
    .groupBy('group_id')
    .agg(
        F.sum('value').alias('value_sum'),
        F.avg('value').alias('value_avg'),
    )
)

spill_candidate_df.explain('formatted')

_, spill_candidate_seconds = run_action(
    'PHASE 8 - EXP 6 - spill candidate',
    spill_candidate_df.count,
)


### Record actual evidence

```text
memory spill:
disk spill:
which tasks spill:
typical task duration:
max task duration:
typical shuffle read:
max shuffle read:
```

### Worked reasoning

If spill is zero:

```text
This workload did not exceed useful in-memory execution capacity in this runtime.
```

If spill is concentrated in one oversized task:

```text
Investigate skew / oversized partition first.
```

If spill is widespread across similarly sized tasks:

```text
Investigate general partition size, operator memory demand, cache pressure,
concurrency, and only then legitimate resource sizing.
```

Therefore non-zero spill does not automatically mean “increase executor memory.”


[Back to Table of Contents](#toc)

---

<a id="experiment-7"></a>
# Experiment 7 — Failed Task Localization

## Question

Can you localize a deterministic failure to the job, stage, task/partition, and expression?

### Worked design

One known row (`id == 7777`) triggers `raise_error()`.

The materializing action must actually use `checked_value`; a plain row count could allow Spark to avoid evaluating an otherwise unused projection.


In [ ]:
failure_demo_df = (
    spark.range(
        start=0,
        end=10000,
        step=1,
        numPartitions=8,
    )
    .select(
        F.col('id'),
        F.when(
            F.col('id') == 7777,
            F.raise_error('intentional Phase 8 failure for id 7777'),
        )
        .otherwise(F.col('id').cast('string'))
        .alias('checked_value'),
    )
)

failure_demo_df.explain('formatted')


In [ ]:
try:
    # max(length(...)) forces evaluation of checked_value while returning only one
    # small aggregate result to the driver.
    run_action(
        'PHASE 8 - EXP 7 - intentional deterministic failure',
        lambda: (
            failure_demo_df
            .agg(F.max(F.length('checked_value')).alias('max_length'))
            .first()
        ),
    )
except Exception as error:
    print('Expected failure captured.')
    print(type(error).__name__)
    print(
        'Inspect the failed job/stage/task now while the Spark application '
        'is still running.'
    )


### Record actual failure evidence

```text
job ID:
stage ID:
failing task / partition:
task attempt(s):
executor:
exception:
same logical task retried?:
```

### Worked interpretation

Repeated failure of the same logical partition with the same deterministic expression error supports a data/logic diagnosis rather than a random capacity diagnosis.

General rule:

```text
same logical partition repeatedly fails similarly
→ deterministic data/logic problem is plausible

failures move broadly across partitions/executors
→ wider resource/infrastructure issue becomes more plausible
```


[Back to Table of Contents](#toc)

---

<a id="experiment-8"></a>
# Experiment 8 — Executor Utilization and Driver Misuse

## Question

Is the bottleneck distributed executor work, limited runnable parallelism, or driver-side behavior?


In [ ]:
executor_work_df = (
    spark.range(
        start=0,
        end=800000,
        step=1,
        numPartitions=32,
    )
    .select(
        F.col('id'),
        (F.col('id') % 10000).alias('group_id'),
    )
    .groupBy('group_id')
    .agg(F.count('*').alias('row_count'))
)

executor_work_df.explain('formatted')

_, executor_work_seconds = run_action(
    'PHASE 8 - EXP 8 - parallel aggregation',
    executor_work_df.count,
)


### Record executor evidence

```text
active/completed tasks:
useful parallelism observed?:
one repeatedly slow/failing executor?:
notable GC?:
notable spill?:
```

### Worked reasoning

Many idle executors do **not** automatically imply insufficient cluster size.

```text
only two runnable tasks
→ partitioning limits parallelism

one or two long stragglers remain
→ most executors naturally become idle near stage completion
```


In [ ]:
driver_safe_diagnostic_df = (
    fact_sales_df
    .groupBy('store_id')
    .agg(
        F.count('*').alias('row_count'),
        F.sum('gross_sales').alias('gross_sales'),
    )
)

driver_safe_rows, driver_safe_seconds = run_action(
    'PHASE 8 - EXP 8 - collect small aggregated diagnostic',
    lambda: driver_safe_diagnostic_df.orderBy('store_id').collect(),
)

print(f'rows collected to driver: {len(driver_safe_rows)}')


### Worked driver-vs-executor distinction

Safe:

```text
large distributed fact
→ aggregate on executors
→ about 40 rows
→ collect small result
```

Risky:

```text
large distributed fact
→ collect entire fact
→ driver memory / Python-local pressure
```

Do **not** intentionally collect an enormous dataset to manufacture a driver OOM.

Increasing executor memory does not repair driver-side `collect()` misuse.


[Back to Table of Contents](#toc)

---

<a id="experiment-9"></a>
# Experiment 9 — AQE Runtime Behavior

## Question

Did AQE actually modify runtime execution?

### Worked prediction

With a high static shuffle-partition target and a small post-shuffle result, AQE **may** coalesce post-shuffle partitions.

That is only a hypothesis until the runtime plan/task counts prove it.


In [ ]:
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '96')

aqe_input_df = (
    spark.range(
        start=0,
        end=240000,
        step=1,
        numPartitions=12,
    )
    .select(
        (F.col('id') % 40).alias('store_bucket'),
        F.col('id').alias('sale_id'),
    )
)

aqe_off_df = (
    aqe_input_df
    .groupBy('store_bucket')
    .agg(F.count('*').alias('row_count'))
)

aqe_off_df.explain('formatted')

_, aqe_off_seconds = run_action(
    'PHASE 8 - EXP 9 - AQE disabled',
    aqe_off_df.count,
)


In [ ]:
# Change ONE thing: enable AQE/coalescing for the same logical workload.
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')

aqe_on_df = (
    aqe_input_df
    .groupBy('store_bucket')
    .agg(F.count('*').alias('row_count'))
)

_, aqe_on_seconds = run_action(
    'PHASE 8 - EXP 9 - AQE enabled',
    aqe_on_df.count,
)

# Inspect after materialization so runtime information can appear in the adaptive plan.
aqe_on_df.explain('formatted')


### Record actual comparison

| Evidence | AQE off | AQE on |
|---|---:|---:|
| Configured shuffle target | 96 | 96 |
| Actual relevant task count |  |  |
| Adaptive/final plan evidence |  |  |
| Coalescing evidence |  |  |
| Stage runtime |  |  |

### Worked reasoning

Valid:

```text
AQE coalesced post-shuffle partitions; the runtime plan/task count proves it.
```

Also valid:

```text
AQE was enabled, but this run showed no material adaptive change.
```

Invalid:

```text
AQE coalesced partitions because spark.sql.adaptive.enabled was true.
```


In [ ]:
# Restore ordinary Phase 8 experiment defaults.
spark.conf.set('spark.sql.shuffle.partitions', '12')
spark.conf.set('spark.sql.adaptive.enabled', 'false')


[Back to Table of Contents](#toc)

---

<a id="experiment-10"></a>
# Experiment 10 — Recomputed Lineage vs. Cache Reuse

## Question

Does repeated runtime evidence justify persistence?

### Worked hypothesis

Two uncached actions over the same expensive join/aggregation lineage can recompute that lineage.

If the intermediate is genuinely reused, caching may trade one materialization/storage cost for cheaper later reuse.


In [ ]:
expensive_reused_df = (
    fact_sales_df
    .filter(F.col('order_status') == 'COMPLETED')
    .join(
        F.broadcast(
            dim_store_df.select(
                'store_id',
                'province',
            )
        ),
        on='store_id',
        how='inner',
    )
    .groupBy(
        'province',
        'product_id',
    )
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

_, uncached_action_1_seconds = run_action(
    'PHASE 8 - EXP 10 - uncached action 1',
    expensive_reused_df.count,
)

_, uncached_action_2_seconds = run_action(
    'PHASE 8 - EXP 10 - uncached action 2',
    lambda: (
        expensive_reused_df
        .agg(F.sum('gross_sales').alias('gross_sales'))
        .first()['gross_sales']
    ),
)


In [ ]:
cached_reused_df = expensive_reused_df.cache()

_, cache_materialization_seconds = run_action(
    'PHASE 8 - EXP 10 - cache materialization',
    cached_reused_df.count,
)

_, cache_reuse_seconds = run_action(
    'PHASE 8 - EXP 10 - cache reuse',
    lambda: (
        cached_reused_df
        .agg(F.sum('gross_sales').alias('gross_sales'))
        .first()['gross_sales']
    ),
)


### Record actual evidence

```text
UNCACHED ACTION 1
- expensive upstream stages:

UNCACHED ACTION 2
- repeated upstream stages?:

CACHE MATERIALIZATION
- materialization cost:
- Storage view cached partitions:

CACHE REUSE
- expensive upstream join/aggregation repeated?:
- new stage structure:
```

### Worked reasoning

Comparing only an uncached first run with a cached reuse run is incomplete because the cached path first paid a materialization cost.

Caching is justified only when:

```text
reuse savings
>
materialization + memory/storage cost
```


In [ ]:
cached_reused_df.unpersist()


[Back to Table of Contents](#toc)

---

<a id="applied-project"></a>
# Applied Phase 8 Project

## Scenario

Diagnose and improve:

```text
partitioned Parquet fact_sales
        ↓
Q1 2026 + COMPLETED filter
        ↓
store dimension join
        ↓
product dimension join
        ↓
province x category aggregation
        ↓
Parquet write
```

Business requirement:

```text
completed Q1 2026 gross sales by province and category
```

Required output grain:

```text
one row per province x category
```

The baseline deliberately forces a sort-merge join to the tiny store dimension.

Your job is not to optimize everything.

Your job is to use runtime evidence to determine the best **first** change.


In [ ]:
# Keep temporary source/output paths alive across notebook cells.
phase_08_temp_dir = TemporaryDirectory()
phase_08_temp_root = Path(phase_08_temp_dir.name)

sales_path = str(phase_08_temp_root / 'fact_sales')
baseline_output_path = str(phase_08_temp_root / 'baseline_output')
optimized_output_path = str(phase_08_temp_root / 'optimized_output')

_, setup_write_seconds = run_action(
    'PHASE 8 - APPLIED - setup partitioned fact sales',
    lambda: (
        fact_sales_df
        .write
        .mode('overwrite')
        .partitionBy(
            'year',
            'month',
        )
        .parquet(sales_path)
    ),
)

source_sales_df = spark.read.parquet(sales_path)


In [ ]:
# Freeze correctness before performance work.
assert_unique_key(dim_store_df, 'store_id')
assert_unique_key(dim_product_df, 'product_id')

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

baseline_pipeline_df = (
    source_sales_df
    .filter(
        (F.col('year') == 2026)
        & (F.col('month') <= 3)
        & (F.col('order_status') == 'COMPLETED')
    )
    .select(
        'store_id',
        'product_id',
        'gross_sales',
    )
    .hint('merge')
    .join(
        dim_store_df
        .select(
            'store_id',
            'province',
        )
        .hint('merge'),
        on='store_id',
        how='inner',
    )
    .join(
        F.broadcast(
            dim_product_df.select(
                'product_id',
                'category',
            )
        ),
        on='product_id',
        how='inner',
    )
    .groupBy(
        'province',
        'category',
    )
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

baseline_pipeline_df.explain('formatted')


In [ ]:
_, applied_baseline_seconds = run_action(
    'PHASE 8 - APPLIED - baseline write',
    lambda: (
        baseline_pipeline_df
        .write
        .mode('overwrite')
        .parquet(baseline_output_path)
    ),
)


## Baseline runtime record

Fill from the actual Spark UI:

```text
ACTION
- labelled write:

JOB / QUERY
- job ID(s):
- query ID:
- wall-clock runtime:

SLOWEST STAGE
- stage ID:
- role:
- task count:
- stage duration:
- input:
- shuffle write:
- shuffle read:
- memory spill:
- disk spill:
- failed/retried tasks:

TASKS
- typical duration:
- max duration:
- typical data size:
- max data size:
- stragglers?:

EXECUTORS
- parallelism:
- repeated slow executor?:

PLAN
- PartitionFilters:
- PushedFilters:
- join 1:
- join 2:
- Exchange(s):
- aggregation:
```

### Worked first-change hypothesis

Because `dim_store_df` is tiny and key-unique, the forced two-sided sort-merge join is intentionally suspicious.

If the plan/UI confirm avoidable join shuffle work, change **only**:

```text
broadcast dim_store_df
```

Deliberately do **not** also change shuffle partitions, AQE, cache state, or memory.


In [ ]:
# Change ONE thing: broadcast the validated-small store dimension.
optimized_pipeline_df = (
    source_sales_df
    .filter(
        (F.col('year') == 2026)
        & (F.col('month') <= 3)
        & (F.col('order_status') == 'COMPLETED')
    )
    .select(
        'store_id',
        'product_id',
        'gross_sales',
    )
    .join(
        F.broadcast(
            dim_store_df.select(
                'store_id',
                'province',
            )
        ),
        on='store_id',
        how='inner',
    )
    .join(
        F.broadcast(
            dim_product_df.select(
                'product_id',
                'category',
            )
        ),
        on='product_id',
        how='inner',
    )
    .groupBy(
        'province',
        'category',
    )
    .agg(F.sum('gross_sales').alias('gross_sales'))
)

optimized_pipeline_df.explain('formatted')


In [ ]:
_, applied_optimized_seconds = run_action(
    'PHASE 8 - APPLIED - broadcast store write',
    lambda: (
        optimized_pipeline_df
        .write
        .mode('overwrite')
        .parquet(optimized_output_path)
    ),
)


In [ ]:
# Reconcile total business measure.
reconcile_scalar(
    baseline_pipeline_df,
    optimized_pipeline_df,
    'gross_sales',
    'applied baseline vs. optimized gross sales',
)

# Reconcile output grain: one row per province x category.
baseline_duplicate_groups = (
    baseline_pipeline_df
    .groupBy(
        'province',
        'category',
    )
    .count()
    .filter(F.col('count') > 1)
    .count()
)

optimized_duplicate_groups = (
    optimized_pipeline_df
    .groupBy(
        'province',
        'category',
    )
    .count()
    .filter(F.col('count') > 1)
    .count()
)

assert baseline_duplicate_groups == 0
assert optimized_duplicate_groups == 0

# The grouped result is small enough for exact deterministic comparison.
baseline_rows = (
    baseline_pipeline_df
    .orderBy(
        'province',
        'category',
    )
    .collect()
)

optimized_rows = (
    optimized_pipeline_df
    .orderBy(
        'province',
        'category',
    )
    .collect()
)

assert baseline_rows == optimized_rows

print('Applied project correctness reconciliation passed.')


## Applied before/after evidence

Fill from the actual UI:

| Evidence | Baseline | Optimized |
|---|---:|---:|
| Join strategy for store dimension |  |  |
| Join-related Exchange(s) |  |  |
| Relevant stage runtime |  |  |
| Task count |  |  |
| Typical task duration |  |  |
| Maximum task duration |  |  |
| Shuffle write |  |  |
| Shuffle read |  |  |
| Memory spill |  |  |
| Disk spill |  |  |
| Failed/retried tasks |  |  |
| Wall-clock runtime |  |  |

## Worked final-diagnosis structure

Use your actual runtime values:

```text
Symptom:
The pipeline was slow.

Action:
The labelled Parquet write triggered the workload.

Location:
Job ___ / stage ___ dominated runtime.

Task evidence:
___

Physical-plan evidence:
___

Root cause:
___

First change:
Broadcast the validated-small store dimension
ONLY if the plan/UI confirm avoidable sort-merge shuffle work.

Why this change first:
___

Rerun evidence:
___

Correctness:
Both versions preserved province x category grain and reconciled exactly.

Next investigation if needed:
___
```

The Phase 8 standard is causal explanation, not metric observation alone.


In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10485760')


[Back to Table of Contents](#toc)

---

<a id="cleanup"></a>
# Cleanup

Run this only after you are finished inspecting the live Spark UI.

Stopping Spark removes the live application UI.


In [ ]:
# Remove temporary Parquet data created only for the applied experiment.
phase_08_temp_dir.cleanup()

# Stop Spark only when all UI inspection is finished.
spark.stop()


## Final Phase 8 diagnostic model

```text
symptom
→ action
→ job
→ stage
→ tasks
→ actual runtime evidence
→ physical plan
→ root cause
→ change ONE thing
→ rerun
→ compare
→ validate correctness
```

Core mastery requirement:

> **Explain why a Spark job is slow rather than merely observe that it is slow.**

[Back to Table of Contents](#toc)
